In [1]:
import pandas as pd
import numpy as np
import os

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Bowler analysis notebook ready!")

Pandas: 3.1.1
NumPy: 2.5.4
Bowler analysis notebook ready!


In [2]:
DATA_PATH = "../outputs/cleaned_data/"
PRESSURE_PATH = "../outputs/pressure_analysis/"
OUTPUT_PATH = "../outputs/bowler_analysis/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

ball_by_ball = pd.read_csv(DATA_PATH + "ball_by_ball_cleaned.csv")
pressure = pd.read_csv(PRESSURE_PATH + "ipl_chase_pressure_deliveries.csv")

print("Ball-by-ball shape:", ball_by_ball.shape)
print("IPL pressure dataset shape:", pressure.shape)

print("\nBall-by-ball columns:")
print(ball_by_ball.columns.tolist())

Ball-by-ball shape: (200999, 33)
IPL pressure dataset shape: (41700, 51)

Ball-by-ball columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']


In [3]:
print("Bowler IDs missing:", ball_by_ball["bowler_id"].isna().sum())

print("\nBowler names missing:", ball_by_ball["bowler_name"].isna().sum())

print("\nDismissal types:")
print(ball_by_ball["dismissal_type"].value_counts(dropna=False))

print("\nExtras types:")
print(ball_by_ball["extras_type"].value_counts(dropna=False))

print("\nUnique bowlers:", ball_by_ball["bowler_id"].nunique())

Bowler IDs missing: 0

Bowler names missing: 0

Dismissal types:
dismissal_type
NaN                      190118
Caught                     7557
Bowled                     1775
LBW                         709
RunOut                      600
Stump                       214
Hit Wicket                   17
Retire Out                    6
Obstracting the field         2
Obstructing the field         1
Name: count, dtype: int64

Extras types:
extras_type
NaN         189167
wide          7714
legbyes       2748
Overstep       576
byes           529
other          264
Penalty          1
Name: count, dtype: int64

Unique bowlers: 647


In [4]:
bowler_credited_dismissals = [
    "Caught",
    "Bowled",
    "LBW",
    "Stump",
    "Hit Wicket"
]

ball_by_ball["is_bowler_credited_wicket"] = (
    ball_by_ball["is_wicket"].eq(True)
    & ball_by_ball["dismissal_type"].isin(bowler_credited_dismissals)
)

print("Total wicket events:",
      ball_by_ball["is_wicket"].sum())

print("Bowler-credited wickets:",
      ball_by_ball["is_bowler_credited_wicket"].sum())

print("\nBowler-credited dismissal breakdown:")
print(
    ball_by_ball.loc[
        ball_by_ball["is_bowler_credited_wicket"],
        "dismissal_type"
    ].value_counts()
)

Total wicket events: 10881
Bowler-credited wickets: 10272

Bowler-credited dismissal breakdown:
dismissal_type
Caught        7557
Bowled        1775
LBW            709
Stump          214
Hit Wicket      17
Name: count, dtype: int64


In [5]:
print("Extras breakdown:")

extras_summary = (
    ball_by_ball.groupby("extras_type", dropna=False)
    .agg(
        delivery_records=("match_id", "size"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        batter_runs=("runs", "sum")
    )
    .reset_index()
)

print(extras_summary.to_string(index=False))

Extras breakdown:
extras_type  delivery_records  extras1_runs  extras2_runs  batter_runs
   Overstep               576           576            48          852
    Penalty                 1             5             0            0
       byes               529          1000             0            0
    legbyes              2748          3603             0            1
      other               264           264            35          412
       wide              7714          9174             0            0
        NaN            189167             1             1       268285


In [6]:
unusual_extras = ball_by_ball[
    (
        ball_by_ball["extras_type"].isna()
        & (
            ball_by_ball["extras1_runs"].fillna(0).ne(0)
            | ball_by_ball["extras_2_runs"].fillna(0).ne(0)
        )
    )
    | (
        ball_by_ball["extras_type"].eq("other")
        & ball_by_ball["extras_2_runs"].fillna(0).ne(0)
    )
]

print("Unusual extras records:", len(unusual_extras))

print(
    unusual_extras[
        [
            "match_id", "innings", "overid", "over_ball_num",
            "striker_name", "bowler_name", "runs",
            "extras_type", "extras1_runs", "extras_2_runs"
        ]
    ].head(25).to_string(index=False)
)

Unusual extras records: 16
 match_id  innings  overid  over_ball_num          striker_name         bowler_name  runs extras_type  extras1_runs  extras_2_runs
    13614        1      18              5         Dasun Shanaka      Mitchell Starc     0       other             1              4
    14127        2       9              3          Prithvi Shaw          Obed McCoy     0       other             1              4
    14159        2       9              1          Ishan Kishan         Umran Malik     0       other             1              1
    17555        1       5              4           Dawid Malan          Haris Rauf     0         NaN             0              1
    20188        1      19              4       M Shahrukh Khan             KM Asif     0       other             1              1
    20234        2      13              3         Nehal Wadhera       Harshal Patel     0       other             1              4
    20243        2      19              4             Ti

In [7]:
print("Unusual extras — delivery details")

print(
    unusual_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "bowler_name",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "delivery_style",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

print("\nDelivery styles for unusual extras:")
print(unusual_extras["delivery_style"].value_counts(dropna=False))

Unusual extras — delivery details
 match_id  innings  overid  over_ball_num         bowler_name extras_type  extras1_runs  extras_2_runs delivery_style  is_wicket dismissal_type
    13614        1      18              5      Mitchell Starc       other             1              4         Beamer      False            NaN
    14127        2       9              3          Obed McCoy       other             1              4   Back of Hand      False            NaN
    14159        2       9              1         Umran Malik       other             1              1         Beamer      False            NaN
    17555        1       5              4          Haris Rauf         NaN             0              1      Seamed In      False            NaN
    20188        1      19              4             KM Asif       other             1              1      Full Toss      False            NaN
    20234        2      13              3       Harshal Patel       other             1              4

In [8]:
print("Nonzero extras_2_runs by extras type:")

extras2_summary = (
    ball_by_ball[
        ball_by_ball["extras_2_runs"].fillna(0) > 0
    ]
    .groupby("extras_type", dropna=False)
    .agg(
        records=("match_id", "size"),
        extra2_total=("extras_2_runs", "sum"),
        extra2_values=("extras_2_runs", "nunique")
    )
    .reset_index()
)

print(extras2_summary.to_string(index=False))

print("\nIndividual extras_2_runs values:")
print(
    ball_by_ball["extras_2_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

Nonzero extras_2_runs by extras type:
extras_type  records  extra2_total  extra2_values
   Overstep       21            48              2
      other       14            35              2
        NaN        1             1              1

Individual extras_2_runs values:
extras_2_runs
0    200963
1        20
4        16


In [9]:
print("Extras column combinations:")

extras_combinations = (
    ball_by_ball.groupby(
        ["extras_type", "extras1_runs", "extras_2_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

print(
    extras_combinations[
        (extras_combinations["extras1_runs"] > 0) |
        (extras_combinations["extras_2_runs"] > 0)
    ].to_string(index=False)
)

Extras column combinations:
extras_type  extras1_runs  extras_2_runs  records
   Overstep             1              0      555
   Overstep             1              1       12
   Overstep             1              4        9
    Penalty             5              0        1
       byes             1              0      348
       byes             2              0       35
       byes             3              0        3
       byes             4              0      142
       byes             5              0        1
    legbyes             1              0     2376
    legbyes             2              0      127
    legbyes             3              0        5
    legbyes             4              0      237
    legbyes             5              0        2
      other             1              0      250
      other             1              1        7
      other             1              4        7
       wide             1              0     7107
       wide           

In [10]:
# Provisional bowler-conceded runs.
# extras_2_runs is excluded pending clarification of its meaning.

bowler_chargeable_extras = ["wide", "Overstep", "other"]

ball_by_ball["bowler_conceded_runs"] = (
    ball_by_ball["runs"].fillna(0)
    + np.where(
        ball_by_ball["extras_type"].isin(bowler_chargeable_extras),
        ball_by_ball["extras1_runs"].fillna(0),
        0
    )
)

print("Total batter runs:",
      ball_by_ball["runs"].sum())

print("Total bowler-conceded runs:",
      ball_by_ball["bowler_conceded_runs"].sum())

print("\nBowler-conceded runs by extras type:")
print(
    ball_by_ball.groupby("extras_type", dropna=False)[
        "bowler_conceded_runs"
    ].sum().to_string()
)

Total batter runs: 269550
Total bowler-conceded runs: 279564

Bowler-conceded runs by extras type:
extras_type
Overstep      1428
Penalty          0
byes             0
legbyes          1
other          676
wide          9174
NaN         268285


In [11]:
print("Leg-bye records with batter runs or unusual extras:")

print(
    ball_by_ball[
        (ball_by_ball["extras_type"] == "legbyes")
        & (
            (ball_by_ball["runs"] > 0)
            | (
                ball_by_ball["extras1_runs"].fillna(0)
                == 0
            )
        )
    ][
        [
            "match_id",
            "innings",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].to_string(index=False)
)

Leg-bye records with batter runs or unusual extras:
 match_id  innings    striker_name bowler_name  runs extras_type  extras1_runs  extras_2_runs
    37099        1 Nimesh Vimukthi  Qais Ahmad     1     legbyes             0              0


In [12]:
# Provisional bowler-conceded runs.
# Leg-byes, byes and penalty runs are excluded.
# extras_2_runs is excluded pending source clarification.

bowler_chargeable_extras = ["wide", "Overstep", "other"]

ball_by_ball["bowler_conceded_runs"] = (
    ball_by_ball["runs"].fillna(0)
    .where(~ball_by_ball["extras_type"].isin(["byes", "legbyes", "Penalty"]), 0)
    + np.where(
        ball_by_ball["extras_type"].isin(bowler_chargeable_extras),
        ball_by_ball["extras1_runs"].fillna(0),
        0
    )
)

print("Total bowler-conceded runs:",
      ball_by_ball["bowler_conceded_runs"].sum())

print("Leg-bye anomaly excluded:",
      ball_by_ball.loc[
          ball_by_ball["match_id"] == 37099,
          "bowler_conceded_runs"
      ].sum())

Total bowler-conceded runs: 279563
Leg-bye anomaly excluded: 260


In [13]:
# Provisional bowler-conceded runs.
# Exclude the identified incorrectly recorded leg-bye batter run.
# extras_2_runs remains excluded pending source clarification.

bowler_chargeable_extras = ["wide", "Overstep", "other"]

legbye_anomaly = (
    ball_by_ball["extras_type"].eq("legbyes")
    & ball_by_ball["runs"].eq(1)
    & ball_by_ball["extras1_runs"].fillna(0).eq(0)
    & ball_by_ball["extras_2_runs"].fillna(0).eq(0)
)

batter_runs_for_bowler = ball_by_ball["runs"].fillna(0).copy()
batter_runs_for_bowler.loc[legbye_anomaly] = 0

ball_by_ball["bowler_conceded_runs"] = (
    batter_runs_for_bowler
    + np.where(
        ball_by_ball["extras_type"].isin(bowler_chargeable_extras),
        ball_by_ball["extras1_runs"].fillna(0),
        0
    )
)

print("Total bowler-conceded runs:",
      ball_by_ball["bowler_conceded_runs"].sum())

print("Anomalous leg-bye records excluded:",
      legbye_anomaly.sum())

print("Runs excluded from bowler totals:",
      batter_runs_for_bowler.loc[legbye_anomaly].sum())

Total bowler-conceded runs: 279563
Anomalous leg-bye records excluded: 1
Runs excluded from bowler totals: 0


In [14]:
# Provisional legal-ball classification

illegal_extras = ["wide", "Overstep", "other"]

ball_by_ball["is_legal_ball"] = (
    ~ball_by_ball["extras_type"].isin(illegal_extras)
)

print("Total delivery records:", len(ball_by_ball))

print("Legal deliveries:",
      ball_by_ball["is_legal_ball"].sum())

print("Illegal deliveries:",
      (~ball_by_ball["is_legal_ball"]).sum())

print("\nLegal-ball classification by extras type:")

print(
    ball_by_ball.groupby("extras_type", dropna=False)[
        "is_legal_ball"
    ].value_counts().to_string()
)

Total delivery records: 200999
Legal deliveries: 192445
Illegal deliveries: 8554

Legal-ball classification by extras type:
extras_type  is_legal_ball
Overstep     False               576
Penalty      True                  1
byes         True                529
legbyes      True               2748
other        False               264
wide         False              7714
NaN          True             189167


In [15]:
over_validation = (
    ball_by_ball
    .groupby(["match_id", "innings", "overid"])
    .agg(
        legal_balls=("is_legal_ball", "sum"),
        delivery_records=("match_id", "size")
    )
    .reset_index()
)

print("Overs with more than 6 legal balls:",
      (over_validation["legal_balls"] > 6).sum())

print("\nMaximum legal balls in an over:",
      over_validation["legal_balls"].max())

print("\nLegal balls per over distribution:")
print(
    over_validation["legal_balls"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nOvers with more than 6 legal balls:")
print(
    over_validation[
        over_validation["legal_balls"] > 6
    ].to_string(index=False)
)

Overs with more than 6 legal balls: 1

Maximum legal balls in an over: 7

Legal balls per over distribution:
legal_balls
0        1
1      102
2      100
3      109
4      133
5      147
6    31757
7        1

Overs with more than 6 legal balls:
 match_id  innings  overid  legal_balls  delivery_records
    41509        1      12            7                 7


In [16]:
unusual_over = ball_by_ball[
    (ball_by_ball["match_id"] == 41509)
    & (ball_by_ball["innings"] == 1)
    & (ball_by_ball["overid"] == 12)
]

print(
    unusual_over[
        [
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "delivery_style",
            "is_legal_ball"
        ]
    ].to_string(index=False)
)

 over_ball_num    striker_name  bowler_name  runs extras_type  extras1_runs  extras_2_runs delivery_style  is_legal_ball
             1 Shimron Hetmyer Aaron Hardie     0         NaN             0              0    Wide Yorker           True
             2 Shimron Hetmyer Aaron Hardie     1         NaN             0              0       Straight           True
             3    Jason Holder Aaron Hardie     6         NaN             0              0      Angled In           True
             4    Jason Holder Aaron Hardie     4         NaN             0              0      Full Toss           True
             5    Jason Holder Aaron Hardie     0         NaN             0              0   Slow Bouncer           True
             6    Jason Holder Aaron Hardie     1         NaN             0              0     Seamed Out           True
             7 Shimron Hetmyer Aaron Hardie     2         NaN             0              0      Full Toss           True


In [17]:
matches = pd.read_csv(DATA_PATH + "matches_cleaned.csv")

match_details = matches[
    matches["match_id"] == 41509
]

print(
    match_details[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "venue_country",
            "result"
        ]
    ].to_string(index=False)
)

 match_id match_date   home_team away_team venue_country                     result
    41509 2025-07-28 West Indies Australia   West Indies Australia Won by 3 wickets


In [18]:
legal_ball_anomalies = over_validation[
    over_validation["legal_balls"] > 6
].copy()

legal_ball_anomalies["anomaly_reason"] = (
    "More than 6 legal deliveries recorded in one over; "
    "source data does not identify an illegal delivery"
)

legal_ball_anomalies["match_description"] = (
    "West Indies vs Australia, 2025-07-28"
)

print("Legal-ball anomalies:")
print(legal_ball_anomalies.to_string(index=False))

Legal-ball anomalies:
 match_id  innings  overid  legal_balls  delivery_records                                                                                       anomaly_reason                    match_description
    41509        1      12            7                 7 More than 6 legal deliveries recorded in one over; source data does not identify an illegal delivery West Indies vs Australia, 2025-07-28


In [19]:
competitions = pd.read_csv(DATA_PATH + "competitions_cleaned.csv")

# Identify IPL competitions
ipl_comp_ids = competitions.loc[
    competitions["comp_name"].str.contains(
        r"\bIPL\b", case=False, na=False
    ),
    "comp_id"
]

# Identify T20I competitions
t20i_comp_ids = competitions.loc[
    competitions["match_format"].eq("T20I"),
    "comp_id"
]

target_comp_ids = list(ipl_comp_ids) + list(t20i_comp_ids)

# Select target matches
target_matches = matches[
    matches["comp_id"].isin(target_comp_ids)
].copy()

# Keep regular innings only (exclude super overs)
target_ball_by_ball = ball_by_ball[
    ball_by_ball["match_id"].isin(target_matches["match_id"])
    & ball_by_ball["innings"].isin([1, 2])
].copy()

print("Target matches:", len(target_matches))
print("Target delivery records:", len(target_ball_by_ball))

print("\nCompetition match formats:")
print(
    target_matches["comp_id"]
    .map(competitions.set_index("comp_id")["match_format"])
    .value_counts()
)

print("\nDelivery records by innings:")
print(target_ball_by_ball["innings"].value_counts().sort_index())

Target matches: 853
Target delivery records: 193988

Competition match formats:
comp_id
T20I    482
T20     371
Name: count, dtype: int64

Delivery records by innings:
innings
1    102054
2     91934
Name: count, dtype: int64


In [20]:
# Recalculate bowling metrics on IPL + T20I regular innings only.

target_ball_by_ball["bowler_conceded_runs"] = (
    target_ball_by_ball["runs"].fillna(0)
    + np.where(
        target_ball_by_ball["extras_type"].isin(
            ["wide", "Overstep", "other"]
        ),
        target_ball_by_ball["extras1_runs"].fillna(0),
        0
    )
)

target_ball_by_ball["is_legal_ball"] = (
    ~target_ball_by_ball["extras_type"].isin(
        ["wide", "Overstep", "other"]
    )
)

target_ball_by_ball["is_bowler_credited_wicket"] = (
    target_ball_by_ball["is_wicket"].eq(True)
    & target_ball_by_ball["dismissal_type"].isin(
        bowler_credited_dismissals
    )
)

print("Target bowler-conceded runs:",
      target_ball_by_ball["bowler_conceded_runs"].sum())

print("Target legal balls:",
      target_ball_by_ball["is_legal_ball"].sum())

print("Target bowler-credited wickets:",
      target_ball_by_ball["is_bowler_credited_wicket"].sum())

print("Unique bowlers:",
      target_ball_by_ball["bowler_id"].nunique())

Target bowler-conceded runs: 270786
Target legal balls: 185692
Target bowler-credited wickets: 9899
Unique bowlers: 596


In [21]:
# Add competition format and season information.

target_ball_by_ball = target_ball_by_ball.merge(
    target_matches[
        ["match_id", "comp_id"]
    ],
    on="match_id",
    how="left"
)

target_ball_by_ball = target_ball_by_ball.merge(
    competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left"
)

print("Dataset shape:", target_ball_by_ball.shape)

print("\nFormat distribution:")
print(target_ball_by_ball["match_format"].value_counts())

print("\nMissing competition formats:",
      target_ball_by_ball["match_format"].isna().sum())

Dataset shape: (193988, 40)

Format distribution:
match_format
T20I    106315
T20      87673
Name: count, dtype: int64

Missing competition formats: 0


In [22]:
# Calculate total innings runs

innings_summary = (
    target_ball_by_ball
    .groupby(["match_id", "innings", "match_format", "season"])
    .agg(
        innings_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        legal_balls=("is_legal_ball", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

innings_summary["total_innings_runs"] = (
    innings_summary["innings_runs"]
    + innings_summary["extras1_runs"]
    + innings_summary["extras2_runs"]
)

print("Innings summary shape:", innings_summary.shape)

print("\nInnings by format:")
print(innings_summary["match_format"].value_counts())

print("\nInnings score statistics:")
print(
    innings_summary["total_innings_runs"]
    .describe()
    .round(2)
)

print("\nInnings count by innings number:")
print(innings_summary["innings"].value_counts().sort_index())

Innings summary shape: (1652, 10)

Innings by format:
match_format
T20I    922
T20     730
Name: count, dtype: int64

Innings score statistics:
count    1652.00
mean      166.67
std        39.89
min         2.00
25%       143.00
50%       167.00
75%       193.00
max       304.00
Name: total_innings_runs, dtype: float64

Innings count by innings number:
innings
1    837
2    815
Name: count, dtype: int64


In [23]:
# Extract first-innings scores
first_innings_scores = innings_summary[
    innings_summary["innings"] == 1
][["match_id", "total_innings_runs"]].copy()

first_innings_scores = first_innings_scores.rename(
    columns={"total_innings_runs": "first_innings_score"}
)

# Build match target table
match_targets = target_matches[
    [
        "match_id",
        "revised_target",
        "first_inn_revised_overs",
        "second_inn_revised_overs"
    ]
].merge(
    first_innings_scores,
    on="match_id",
    how="left"
)

# Default target = first-innings score + 1
match_targets["chase_target"] = (
    match_targets["first_innings_score"] + 1
)

# Apply revised target wherever available
match_targets["chase_target"] = (
    match_targets["revised_target"]
    .fillna(match_targets["chase_target"])
)

print("Match target records:", len(match_targets))

print("Missing first-innings scores:",
      match_targets["first_innings_score"].isna().sum())

print("Missing chase targets:",
      match_targets["chase_target"].isna().sum())

print("\nRevised target records:",
      match_targets["revised_target"].notna().sum())

print("\nRevised innings limits:")
print(
    match_targets[
        [
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ].notna().sum()
)

Match target records: 853
Missing first-innings scores: 16
Missing chase targets: 16

Revised target records: 23

Revised innings limits:
first_inn_revised_overs     25
second_inn_revised_overs    25
dtype: int64


In [24]:
# Merge chase targets into delivery dataset

target_ball_by_ball = target_ball_by_ball.merge(
    match_targets[
        [
            "match_id",
            "chase_target",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="left"
)

# Default innings limit: 120 legal balls
target_ball_by_ball["innings_ball_limit"] = 120

# Apply revised limits in legal balls
first_innings = target_ball_by_ball["innings"] == 1
second_innings = target_ball_by_ball["innings"] == 2

target_ball_by_ball.loc[
    first_innings,
    "innings_ball_limit"
] = (
    target_ball_by_ball.loc[
        first_innings, "first_inn_revised_overs"
    ].fillna(20) * 6
)

target_ball_by_ball.loc[
    second_innings,
    "innings_ball_limit"
] = (
    target_ball_by_ball.loc[
        second_innings, "second_inn_revised_overs"
    ].fillna(20) * 6
)

print("Updated dataset shape:", target_ball_by_ball.shape)

print("\nMissing chase targets in delivery records:",
      target_ball_by_ball["chase_target"].isna().sum())

print("\nInnings ball-limit distribution:")
print(
    target_ball_by_ball.groupby(
        ["innings", "innings_ball_limit"]
    ).size().to_string()
)

Updated dataset shape: (193988, 44)

Missing chase targets in delivery records: 0

Innings ball-limit distribution:
innings  innings_ball_limit
1        120                   100425
         180                       34
         252                       44
         288                       49
         324                      173
         360                       60
         396                      216
         432                       75
         468                       79
         504                      192
         540                      182
         612                      211
         648                      173
         684                      141
2        120                    90293
         180                       63
         252                       82
         288                      138
         360                      124
         396                       72
         432                       25
         468                       62
         504        

In [26]:
print("Revised overs-related columns:")

print([
    col for col in target_ball_by_ball.columns
    if "revised" in col or "innings_ball_limit" in col
])

Revised overs-related columns:
['first_inn_revised_overs_x', 'second_inn_revised_overs_x', 'innings_ball_limit', 'first_inn_revised_overs_y', 'second_inn_revised_overs_y']


In [28]:
print("Target-related columns:")

print([
    col for col in target_ball_by_ball.columns
    if "target" in col or "revised" in col or "innings_ball_limit" in col
])

print("\nDataset shape:", target_ball_by_ball.shape)

print("\nMatch target columns:")
print(match_targets.columns.tolist())

Target-related columns:
['chase_target_x', 'innings_ball_limit', 'chase_target_y']

Dataset shape: (193988, 43)

Match target columns:
['match_id', 'revised_target', 'first_inn_revised_overs', 'second_inn_revised_overs', 'first_innings_score', 'chase_target']


In [30]:
# Rebuild combined IPL + T20I dataset from the original data

target_ball_by_ball = ball_by_ball[
    ball_by_ball["match_id"].isin(target_matches["match_id"])
    & ball_by_ball["innings"].isin([1, 2])
].copy()

# Add competition information
target_ball_by_ball = target_ball_by_ball.merge(
    target_matches[["match_id", "comp_id"]],
    on="match_id",
    how="left"
)

target_ball_by_ball = target_ball_by_ball.merge(
    competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left"
)

# Add chase targets and revised innings limits
target_ball_by_ball = target_ball_by_ball.merge(
    match_targets[
        [
            "match_id",
            "chase_target",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="left"
)

# Default innings limit is 120 legal balls
target_ball_by_ball["innings_ball_limit"] = 120

first_innings = target_ball_by_ball["innings"] == 1
second_innings = target_ball_by_ball["innings"] == 2

target_ball_by_ball.loc[
    first_innings, "innings_ball_limit"
] = target_ball_by_ball.loc[
    first_innings, "first_inn_revised_overs"
].fillna(120)

target_ball_by_ball.loc[
    second_innings, "innings_ball_limit"
] = target_ball_by_ball.loc[
    second_innings, "second_inn_revised_overs"
].fillna(120)

print("Rebuilt dataset shape:", target_ball_by_ball.shape)

print("\nMissing chase targets:",
      target_ball_by_ball["chase_target"].isna().sum())

print("\nMissing competition formats:",
      target_ball_by_ball["match_format"].isna().sum())

print("\nInnings ball-limit distribution:")
print(
    target_ball_by_ball.groupby(
        ["innings", "innings_ball_limit"]
    ).size().to_string()
)

print("\nMaximum innings ball limit:",
      target_ball_by_ball["innings_ball_limit"].max())

print("\nDuplicate columns:",
      target_ball_by_ball.columns[
          target_ball_by_ball.columns.duplicated()
      ].tolist())

Rebuilt dataset shape: (193988, 44)

Missing chase targets: 0

Missing competition formats: 0

Innings ball-limit distribution:
innings  innings_ball_limit
1        30                        34
         42                        44
         48                        49
         54                       173
         60                        60
         66                       216
         72                        75
         78                        79
         84                       192
         90                       182
         102                      211
         108                      173
         114                      141
         120                   100425
2        30                        63
         42                        82
         48                       138
         60                       124
         66                        72
         72                        25
         78                        62
         84                        75
        

In [31]:
# Recalculate bowler-conceded runs
# extras_2_runs remains excluded pending clarification.

bowler_chargeable_extras = ["wide", "Overstep", "other"]

target_ball_by_ball["bowler_conceded_runs"] = (
    target_ball_by_ball["runs"].fillna(0)
    + np.where(
        target_ball_by_ball["extras_type"].isin(
            bowler_chargeable_extras
        ),
        target_ball_by_ball["extras1_runs"].fillna(0),
        0
    )
)

# Recalculate legal balls
target_ball_by_ball["is_legal_ball"] = (
    ~target_ball_by_ball["extras_type"].isin(
        ["wide", "Overstep", "other"]
    )
)

# Recalculate bowler-credited wickets
target_ball_by_ball["is_bowler_credited_wicket"] = (
    target_ball_by_ball["is_wicket"].eq(True)
    & target_ball_by_ball["dismissal_type"].isin(
        bowler_credited_dismissals
    )
)

print("Bowler-conceded runs:",
      target_ball_by_ball["bowler_conceded_runs"].sum())

print("Legal balls:",
      target_ball_by_ball["is_legal_ball"].sum())

print("Bowler-credited wickets:",
      target_ball_by_ball["is_bowler_credited_wicket"].sum())

print("Unique bowlers:",
      target_ball_by_ball["bowler_id"].nunique())

Bowler-conceded runs: 270786
Legal balls: 185692
Bowler-credited wickets: 9899
Unique bowlers: 596


In [33]:
# Sort deliveries in match order

target_ball_by_ball = target_ball_by_ball.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

# Total runs scored per delivery
target_ball_by_ball["total_delivery_runs"] = (
    target_ball_by_ball["runs"].fillna(0)
    + target_ball_by_ball["extras1_runs"].fillna(0)
    + target_ball_by_ball["extras_2_runs"].fillna(0)
)

# Convert wicket indicator to integer
target_ball_by_ball["wicket_count"] = (
    target_ball_by_ball["is_wicket"].astype(int)
)

# Group by match and innings
innings_groups = target_ball_by_ball.groupby(
    ["match_id", "innings"],
    sort=False
)

# Calculate cumulative innings state
target_ball_by_ball["cumulative_runs"] = (
    innings_groups["total_delivery_runs"].cumsum()
)

target_ball_by_ball["cumulative_legal_balls"] = (
    innings_groups["is_legal_ball"].cumsum()
)

target_ball_by_ball["cumulative_wickets"] = (
    innings_groups["wicket_count"].cumsum()
)

target_ball_by_ball["wickets_remaining"] = (
    10 - target_ball_by_ball["cumulative_wickets"]
)

print("Negative wickets remaining:",
      (target_ball_by_ball["wickets_remaining"] < 0).sum())

print("Maximum cumulative wickets:",
      target_ball_by_ball["cumulative_wickets"].max())

print("Cumulative innings state calculated successfully.")

Negative wickets remaining: 0
Maximum cumulative wickets: 10
Cumulative innings state calculated successfully.


In [34]:
# Calculate match state before each delivery

target_ball_by_ball["runs_before"] = (
    target_ball_by_ball["cumulative_runs"]
    - target_ball_by_ball["total_delivery_runs"]
)

target_ball_by_ball["legal_balls_before"] = (
    target_ball_by_ball["cumulative_legal_balls"]
    - target_ball_by_ball["is_legal_ball"].astype(int)
)

target_ball_by_ball["wickets_before"] = (
    target_ball_by_ball["cumulative_wickets"]
    - target_ball_by_ball["wicket_count"]
)

# Calculate remaining balls and wickets
target_ball_by_ball["balls_remaining_before"] = (
    target_ball_by_ball["innings_ball_limit"]
    - target_ball_by_ball["legal_balls_before"]
)

target_ball_by_ball["wickets_remaining_before"] = (
    10 - target_ball_by_ball["wickets_before"]
)

print("Negative balls remaining:",
      (target_ball_by_ball["balls_remaining_before"] < 0).sum())

print("Negative wickets remaining:",
      (target_ball_by_ball["wickets_remaining_before"] < 0).sum())

print("\nState before each delivery:")
print(
    target_ball_by_ball[
        [
            "match_id",
            "innings",
            "runs_before",
            "legal_balls_before",
            "wickets_before",
            "balls_remaining_before",
            "wickets_remaining_before"
        ]
    ].head(10)
)

Negative balls remaining: 0
Negative wickets remaining: 0

State before each delivery:
   match_id  innings  runs_before  legal_balls_before  wickets_before  \
0     13511        1            0                   0               0   
1     13511        1            0                   1               0   
2     13511        1            0                   2               0   
3     13511        1            0                   3               0   
4     13511        1            0                   4               0   
5     13511        1            6                   5               0   
6     13511        1            6                   6               1   
7     13511        1            6                   7               1   
8     13511        1            6                   8               1   
9     13511        1            6                   9               1   

   balls_remaining_before  wickets_remaining_before  
0                     120                        10  
1

In [35]:
# Calculate pressure indicators before each delivery

# Wicket pressure
target_ball_by_ball["wicket_pressure"] = (
    (10 - target_ball_by_ball["wickets_remaining_before"]) / 10
) * 100

# Time pressure
target_ball_by_ball["time_pressure"] = (
    1 - (
        target_ball_by_ball["balls_remaining_before"]
        / target_ball_by_ball["innings_ball_limit"]
    )
) * 100

# Runs remaining in second innings
second_innings = target_ball_by_ball["innings"] == 2

target_ball_by_ball["runs_remaining_before"] = np.nan

target_ball_by_ball.loc[
    second_innings, "runs_remaining_before"
] = (
    target_ball_by_ball.loc[second_innings, "chase_target"]
    - target_ball_by_ball.loc[second_innings, "runs_before"]
)

# Required run rate before each delivery
valid_rrr = (
    second_innings
    & (target_ball_by_ball["runs_remaining_before"] > 0)
    & (target_ball_by_ball["balls_remaining_before"] > 0)
)

target_ball_by_ball["required_run_rate_before"] = np.nan

target_ball_by_ball.loc[valid_rrr, "required_run_rate_before"] = (
    target_ball_by_ball.loc[valid_rrr, "runs_remaining_before"]
    / target_ball_by_ball.loc[valid_rrr, "balls_remaining_before"]
) * 6

print("Valid required run rate records:", valid_rrr.sum())

print("\nRequired run rate percentiles:")
print(
    target_ball_by_ball.loc[
        valid_rrr, "required_run_rate_before"
    ].quantile([0.05, 0.50, 0.95])
)

print("\nPressure indicator missing values:")
print(
    target_ball_by_ball[
        ["wicket_pressure", "time_pressure"]
    ].isna().sum()
)

Valid required run rate records: 91922

Required run rate percentiles:
0.05     4.222222
0.50     9.662338
0.95    21.000000
Name: required_run_rate_before, dtype: float64

Pressure indicator missing values:
wicket_pressure    0
time_pressure      0
dtype: int64


In [36]:
# Normalize required run rate pressure using 5th and 95th percentiles

rrr_lower = 4.222222
rrr_upper = 21.0

target_ball_by_ball["rrr_pressure"] = np.nan

target_ball_by_ball.loc[valid_rrr, "rrr_pressure"] = (
    (
        target_ball_by_ball.loc[
            valid_rrr, "required_run_rate_before"
        ].clip(rrr_lower, rrr_upper) - rrr_lower
    ) / (rrr_upper - rrr_lower)
) * 100

print("RRR pressure summary:")
print(target_ball_by_ball["rrr_pressure"].describe())

print("\nRRR pressure range:")
print(
    target_ball_by_ball["rrr_pressure"].min(),
    target_ball_by_ball["rrr_pressure"].max()
)

RRR pressure summary:
count    91922.000000
mean        36.260590
std         24.193989
min          0.000000
25%         20.612615
50%         32.424530
75%         46.357617
max        100.000000
Name: rrr_pressure, dtype: float64

RRR pressure range:
0.0 100.0


In [37]:
# Calculate combined pressure score

first_innings = target_ball_by_ball["innings"] == 1
second_innings = target_ball_by_ball["innings"] == 2

target_ball_by_ball["pressure_score"] = np.nan

# First innings: wicket pressure + time pressure
target_ball_by_ball.loc[first_innings, "pressure_score"] = (
    target_ball_by_ball.loc[first_innings, "wicket_pressure"]
    + target_ball_by_ball.loc[first_innings, "time_pressure"]
) / 2

# Second innings: RRR + wicket + time pressure
target_ball_by_ball.loc[second_innings, "pressure_score"] = (
    target_ball_by_ball.loc[second_innings, "rrr_pressure"]
    + target_ball_by_ball.loc[second_innings, "wicket_pressure"]
    + target_ball_by_ball.loc[second_innings, "time_pressure"]
) / 3

print("Pressure score summary:")
print(target_ball_by_ball["pressure_score"].describe())

print("\nMissing pressure scores:",
      target_ball_by_ball["pressure_score"].isna().sum())

print("\nPressure score range:",
      target_ball_by_ball["pressure_score"].min(),
      target_ball_by_ball["pressure_score"].max())

Pressure score summary:
count    193976.000000
mean         37.253821
std          22.458830
min           0.000000
25%          19.166667
50%          34.323105
75%          52.942644
max          96.388889
Name: pressure_score, dtype: float64

Missing pressure scores: 12

Pressure score range: 0.0 96.3888888888889


In [38]:
# Classify delivery pressure levels

pressure_conditions = [
    target_ball_by_ball["pressure_score"].between(0, 25, inclusive="left"),
    target_ball_by_ball["pressure_score"].between(25, 50, inclusive="left"),
    target_ball_by_ball["pressure_score"].between(50, 75, inclusive="left"),
    target_ball_by_ball["pressure_score"].between(75, 100, inclusive="both")
]

pressure_labels = [
    "Low",
    "Moderate",
    "High",
    "Extreme"
]

target_ball_by_ball["pressure_category"] = np.select(
    pressure_conditions,
    pressure_labels,
    default="Excluded"
)

# Combine High and Extreme as high pressure
target_ball_by_ball["pressure_group"] = np.where(
    target_ball_by_ball["pressure_category"].isin(
        ["High", "Extreme"]
    ),
    "High",
    "Normal"
)

print("Pressure category distribution:")
print(target_ball_by_ball["pressure_category"].value_counts())

print("\nPressure group distribution:")
print(target_ball_by_ball["pressure_group"].value_counts())

print("\nExcluded deliveries:",
      (target_ball_by_ball["pressure_category"] == "Excluded").sum())

Pressure category distribution:
pressure_category
Moderate    71515
Low         67078
High        42185
Extreme     13198
Excluded       12
Name: count, dtype: int64

Pressure group distribution:
pressure_group
Normal    138605
High       55383
Name: count, dtype: int64

Excluded deliveries: 12


In [40]:
# Assign pressure groups without including excluded deliveries

target_ball_by_ball["pressure_group"] = pd.Series(
    index=target_ball_by_ball.index,
    dtype="object"
)

target_ball_by_ball.loc[
    target_ball_by_ball["pressure_category"].isin(["Low", "Moderate"]),
    "pressure_group"
] = "Normal"

target_ball_by_ball.loc[
    target_ball_by_ball["pressure_category"].isin(["High", "Extreme"]),
    "pressure_group"
] = "High"

print("Pressure group distribution:")
print(
    target_ball_by_ball["pressure_group"].value_counts(dropna=False)
)

print("\nPressure category distribution:")
print(target_ball_by_ball["pressure_category"].value_counts())

print("\nPressure indicator averages:")
print(
    target_ball_by_ball[
        ["wicket_pressure", "time_pressure",
         "rrr_pressure", "pressure_score"]
    ].mean()
)

Pressure group distribution:
pressure_group
Normal    138593
High       55383
NaN           12
Name: count, dtype: int64

Pressure category distribution:
pressure_category
Moderate    71515
Low         67078
High        42185
Extreme     13198
Excluded       12
Name: count, dtype: int64

Pressure indicator averages:
wicket_pressure    26.781090
time_pressure      47.854778
rrr_pressure       36.260590
pressure_score     37.253821
dtype: float64


In [41]:
# Compare pressure categories by innings

print(
    pd.crosstab(
        target_ball_by_ball["innings"],
        target_ball_by_ball["pressure_category"]
    )
)

print("\nPressure score boundary counts:")

for boundary in [25, 50, 75]:
    print(
        f"Exactly {boundary}:",
        (target_ball_by_ball["pressure_score"] == boundary).sum()
    )

print("\nInnings ball-limit distribution:")
print(
    target_ball_by_ball.groupby(
        ["innings", "innings_ball_limit"]
    ).size()
)

pressure_category  Excluded  Extreme   High    Low  Moderate
innings                                                     
1                         0     6618  27657  35041     32738
2                        12     6580  14528  32037     38777

Pressure score boundary counts:
Exactly 25: 595
Exactly 50: 533
Exactly 75: 240

Innings ball-limit distribution:
innings  innings_ball_limit
1        30                        34
         42                        44
         48                        49
         54                       173
         60                        60
         66                       216
         72                        75
         78                        79
         84                       192
         90                       182
         102                      211
         108                      173
         114                      141
         120                   100425
2        30                        63
         42                        82
     

In [42]:
# Match the exact pressure classification used in Notebook05

target_ball_by_ball["pressure_category"] = pd.cut(
    target_ball_by_ball["pressure_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Low", "Moderate", "High", "Extreme"]
)

# Assign Normal and High pressure groups
target_ball_by_ball["pressure_group"] = (
    target_ball_by_ball["pressure_category"]
    .map({
        "Low": "Normal",
        "Moderate": "Normal",
        "High": "High",
        "Extreme": "High"
    })
)

print("Pressure category distribution:")
print(
    target_ball_by_ball["pressure_category"]
    .value_counts()
    .sort_index()
)

print("\nPressure group distribution:")
print(
    target_ball_by_ball["pressure_group"]
    .value_counts(dropna=False)
)

print("\nExcluded deliveries:",
      target_ball_by_ball["pressure_score"].isna().sum())

Pressure category distribution:
pressure_category
Low         67673
Moderate    71453
High        41892
Extreme     12958
Name: count, dtype: int64

Pressure group distribution:
pressure_group
Normal    139126
High       54850
NaN           12
Name: count, dtype: int64

Excluded deliveries: 12


In [43]:
# Recalculate exact RRR normalization boundaries
# Same method as Notebook05

rrr_lower = target_ball_by_ball.loc[
    valid_rrr, "required_run_rate_before"
].quantile(0.05)

rrr_upper = target_ball_by_ball.loc[
    valid_rrr, "required_run_rate_before"
].quantile(0.95)

# Recalculate normalized RRR pressure
target_ball_by_ball["rrr_pressure"] = np.nan

target_ball_by_ball.loc[valid_rrr, "rrr_pressure"] = (
    (
        target_ball_by_ball.loc[
            valid_rrr, "required_run_rate_before"
        ].clip(lower=rrr_lower, upper=rrr_upper)
        - rrr_lower
    ) / (rrr_upper - rrr_lower)
) * 100

print("Exact 5th percentile:", repr(rrr_lower))
print("Exact 95th percentile:", repr(rrr_upper))

print("\nRRR pressure summary:")
print(target_ball_by_ball["rrr_pressure"].describe())

Exact 5th percentile: np.float64(4.222222222222222)
Exact 95th percentile: np.float64(21.0)

RRR pressure summary:
count    91922.000000
mean        36.260590
std         24.193989
min          0.000000
25%         20.612614
50%         32.424529
75%         46.357616
max        100.000000
Name: rrr_pressure, dtype: float64


In [44]:
# Recalculate combined pressure scores

first_innings = target_ball_by_ball["innings"] == 1
second_innings = target_ball_by_ball["innings"] == 2

target_ball_by_ball["pressure_score"] = np.nan

target_ball_by_ball.loc[first_innings, "pressure_score"] = (
    target_ball_by_ball.loc[first_innings, "wicket_pressure"]
    + target_ball_by_ball.loc[first_innings, "time_pressure"]
) / 2

target_ball_by_ball.loc[second_innings, "pressure_score"] = (
    target_ball_by_ball.loc[second_innings, "rrr_pressure"]
    + target_ball_by_ball.loc[second_innings, "wicket_pressure"]
    + target_ball_by_ball.loc[second_innings, "time_pressure"]
) / 3

# Apply exactly the same category boundaries as Notebook05
target_ball_by_ball["pressure_category"] = pd.cut(
    target_ball_by_ball["pressure_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Low", "Moderate", "High", "Extreme"]
)

target_ball_by_ball["pressure_group"] = (
    target_ball_by_ball["pressure_category"]
    .map({
        "Low": "Normal",
        "Moderate": "Normal",
        "High": "High",
        "Extreme": "High"
    })
)

print("Pressure category distribution:")
print(target_ball_by_ball["pressure_category"].value_counts().sort_index())

print("\nPressure group distribution:")
print(target_ball_by_ball["pressure_group"].value_counts(dropna=False))

print("\nMissing pressure scores:",
      target_ball_by_ball["pressure_score"].isna().sum())

Pressure category distribution:
pressure_category
Low         67678
Moderate    71448
High        41892
Extreme     12958
Name: count, dtype: int64

Pressure group distribution:
pressure_group
Normal    139126
High       54850
NaN           12
Name: count, dtype: int64

Missing pressure scores: 12


In [45]:
# Keep only deliveries with valid pressure classifications

active_pressure_deliveries = target_ball_by_ball[
    target_ball_by_ball["pressure_group"].notna()
].copy()

# Identify legal dot balls from the bowler's perspective
active_pressure_deliveries["is_dot_ball"] = (
    active_pressure_deliveries["is_legal_ball"]
    & active_pressure_deliveries["bowler_conceded_runs"].eq(0)
)

# Identify boundaries conceded off the bat
active_pressure_deliveries["is_boundary_conceded"] = (
    active_pressure_deliveries["is_four"].eq(True)
    | active_pressure_deliveries["is_six"].eq(True)
)

print("Active pressure deliveries:",
      len(active_pressure_deliveries))

print("\nPressure group distribution:")
print(active_pressure_deliveries["pressure_group"].value_counts())

print("\nUnique bowlers:",
      active_pressure_deliveries["bowler_id"].nunique())

print("\nMissing bowler IDs:",
      active_pressure_deliveries["bowler_id"].isna().sum())

Active pressure deliveries: 193976

Pressure group distribution:
pressure_group
Normal    139126
High       54850
Name: count, dtype: int64

Unique bowlers: 596

Missing bowler IDs: 0


In [46]:
# Aggregate bowling performance by bowler and pressure group

bowler_pressure_stats = (
    active_pressure_deliveries
    .groupby(
        ["bowler_id", "bowler_name", "pressure_group"]
    )
    .agg(
        total_deliveries=("match_id", "size"),
        legal_balls=("is_legal_ball", "sum"),
        runs_conceded=("bowler_conceded_runs", "sum"),
        wickets=("is_bowler_credited_wicket", "sum"),
        matches=("match_id", "nunique"),
        dot_balls=("is_dot_ball", "sum"),
        boundaries_conceded=("is_boundary_conceded", "sum")
    )
    .reset_index()
)

# Calculate bowling performance metrics
bowler_pressure_stats["economy_rate"] = (
    bowler_pressure_stats["runs_conceded"]
    / bowler_pressure_stats["legal_balls"]
) * 6

bowler_pressure_stats["bowling_average"] = (
    bowler_pressure_stats["runs_conceded"]
    / bowler_pressure_stats["wickets"].replace(0, np.nan)
)

bowler_pressure_stats["bowling_strike_rate"] = (
    bowler_pressure_stats["legal_balls"]
    / bowler_pressure_stats["wickets"].replace(0, np.nan)
)

bowler_pressure_stats["dot_ball_percentage"] = (
    bowler_pressure_stats["dot_balls"]
    / bowler_pressure_stats["legal_balls"]
) * 100

bowler_pressure_stats["boundary_percentage"] = (
    bowler_pressure_stats["boundaries_conceded"]
    / bowler_pressure_stats["legal_balls"]
) * 100

print("Bowler pressure statistics shape:",
      bowler_pressure_stats.shape)

print("\nSample records:")
print(bowler_pressure_stats.head(10).to_string(index=False))

Bowler pressure statistics shape: (1106, 15)

Sample records:
 bowler_id     bowler_name pressure_group  total_deliveries  legal_balls  runs_conceded  wickets  matches  dot_balls  boundaries_conceded  economy_rate  bowling_average  bowling_strike_rate  dot_ball_percentage  boundary_percentage
        23   Piyush Chawla           High                86           83            127        5       12         28                   17      9.180723        25.400000            16.600000            33.734940            20.481928
        23   Piyush Chawla         Normal               500          493            680       30       27        158                   78      8.275862        22.666667            16.433333            32.048682            15.821501
        28   Ishant Sharma           High               128          114            208        9       16         26                   27     10.947368        23.111111            12.666667            22.807018            23.684211
        28

In [47]:
# Inspect high-pressure bowling sample sizes

high_pressure_bowlers = bowler_pressure_stats[
    bowler_pressure_stats["pressure_group"] == "High"
].copy()

print("High-pressure bowler sample distribution:")

print(
    high_pressure_bowlers[
        ["legal_balls", "matches", "wickets"]
    ].describe()
)

print("\nEligible bowlers at different sample thresholds:")

for min_balls, min_matches in [
    (30, 3),
    (50, 5),
    (75, 5),
    (100, 10),
    (150, 10)
]:
    eligible = high_pressure_bowlers[
        (high_pressure_bowlers["legal_balls"] >= min_balls)
        & (high_pressure_bowlers["matches"] >= min_matches)
    ]

    print(
        f"At least {min_balls} high-pressure balls "
        f"and {min_matches} matches: {len(eligible)} bowlers"
    )

High-pressure bowler sample distribution:
       legal_balls     matches     wickets
count   518.000000  518.000000  518.000000
mean    100.040541   11.515444    7.277992
std     146.319040   15.832377   11.136568
min       1.000000    1.000000    0.000000
25%      13.000000    2.000000    1.000000
50%      35.000000    4.000000    3.000000
75%     123.750000   15.000000    9.000000
max    1170.000000  124.000000   84.000000

Eligible bowlers at different sample thresholds:
At least 30 high-pressure balls and 3 matches: 280 bowlers
At least 50 high-pressure balls and 5 matches: 222 bowlers
At least 75 high-pressure balls and 5 matches: 183 bowlers
At least 100 high-pressure balls and 10 matches: 156 bowlers
At least 150 high-pressure balls and 10 matches: 108 bowlers


In [48]:
# Select bowlers meeting the primary high-pressure threshold

eligible_high_bowlers = high_pressure_bowlers[
    (high_pressure_bowlers["legal_balls"] >= 100)
    & (high_pressure_bowlers["matches"] >= 10)
].copy()

eligible_bowler_ids = eligible_high_bowlers["bowler_id"].unique()

# Retrieve their normal-pressure statistics
normal_pressure_bowlers = bowler_pressure_stats[
    (bowler_pressure_stats["pressure_group"] == "Normal")
    & (bowler_pressure_stats["bowler_id"].isin(eligible_bowler_ids))
].copy()

print("Eligible high-pressure bowlers:",
      len(eligible_high_bowlers))

print("With normal-pressure records:",
      normal_pressure_bowlers["bowler_id"].nunique())

print("\nNormal-pressure sample distribution:")
print(
    normal_pressure_bowlers[
        ["legal_balls", "matches", "wickets"]
    ].describe()
)

print("\nNormal-pressure threshold counts:")

for min_balls, min_matches in [
    (30, 3),
    (50, 5),
    (100, 10)
]:
    eligible = normal_pressure_bowlers[
        (normal_pressure_bowlers["legal_balls"] >= min_balls)
        & (normal_pressure_bowlers["matches"] >= min_matches)
    ]

    print(
        f"At least {min_balls} balls and {min_matches} matches:",
        len(eligible)
    )

Eligible high-pressure bowlers: 156
With normal-pressure records: 156

Normal-pressure sample distribution:
       legal_balls     matches     wickets
count   156.000000  156.000000  156.000000
mean    638.993590   43.833333   29.538462
std     408.562918   25.209019   19.215378
min     109.000000   10.000000    3.000000
25%     314.000000   24.750000   15.000000
50%     548.500000   38.500000   25.000000
75%     829.000000   53.250000   38.250000
max    2140.000000  146.000000  108.000000

Normal-pressure threshold counts:
At least 30 balls and 3 matches: 156
At least 50 balls and 5 matches: 156
At least 100 balls and 10 matches: 156


In [49]:
# Select bowlers meeting the primary high-pressure threshold

eligible_high_bowlers = high_pressure_bowlers[
    (high_pressure_bowlers["legal_balls"] >= 100)
    & (high_pressure_bowlers["matches"] >= 10)
].copy()

eligible_bowler_ids = eligible_high_bowlers["bowler_id"].unique()

# Retrieve their normal-pressure statistics
normal_pressure_bowlers = bowler_pressure_stats[
    (bowler_pressure_stats["pressure_group"] == "Normal")
    & (bowler_pressure_stats["bowler_id"].isin(eligible_bowler_ids))
].copy()

print("Eligible high-pressure bowlers:",
      len(eligible_high_bowlers))

print("With normal-pressure records:",
      normal_pressure_bowlers["bowler_id"].nunique())

print("\nNormal-pressure sample distribution:")
print(
    normal_pressure_bowlers[
        ["legal_balls", "matches", "wickets"]
    ].describe()
)

print("\nNormal-pressure threshold counts:")

for min_balls, min_matches in [
    (30, 3),
    (50, 5),
    (100, 10)
]:
    eligible = normal_pressure_bowlers[
        (normal_pressure_bowlers["legal_balls"] >= min_balls)
        & (normal_pressure_bowlers["matches"] >= min_matches)
    ]

    print(
        f"At least {min_balls} balls and {min_matches} matches:",
        len(eligible)
    )

Eligible high-pressure bowlers: 156
With normal-pressure records: 156

Normal-pressure sample distribution:
       legal_balls     matches     wickets
count   156.000000  156.000000  156.000000
mean    638.993590   43.833333   29.538462
std     408.562918   25.209019   19.215378
min     109.000000   10.000000    3.000000
25%     314.000000   24.750000   15.000000
50%     548.500000   38.500000   25.000000
75%     829.000000   53.250000   38.250000
max    2140.000000  146.000000  108.000000

Normal-pressure threshold counts:
At least 30 balls and 3 matches: 156
At least 50 balls and 5 matches: 156
At least 100 balls and 10 matches: 156


In [50]:
# Combine Normal and High pressure statistics

eligible_bowler_stats = bowler_pressure_stats[
    bowler_pressure_stats["bowler_id"].isin(eligible_bowler_ids)
].copy()

bowler_comparison = eligible_bowler_stats.pivot(
    index=["bowler_id", "bowler_name"],
    columns="pressure_group",
    values=[
        "legal_balls",
        "runs_conceded",
        "wickets",
        "matches",
        "economy_rate",
        "bowling_average",
        "bowling_strike_rate",
        "dot_ball_percentage",
        "boundary_percentage"
    ]
)

# Flatten column names
bowler_comparison.columns = [
    f"{metric}_{pressure.lower()}"
    for metric, pressure in bowler_comparison.columns
]

bowler_comparison = bowler_comparison.reset_index()

print("Comparison dataset shape:", bowler_comparison.shape)

print("\nColumns:")
print(bowler_comparison.columns.tolist())

print("\nSample comparison:")
print(bowler_comparison.head(5).to_string(index=False))

Comparison dataset shape: (156, 20)

Columns:
['bowler_id', 'bowler_name', 'legal_balls_high', 'legal_balls_normal', 'runs_conceded_high', 'runs_conceded_normal', 'wickets_high', 'wickets_normal', 'matches_high', 'matches_normal', 'economy_rate_high', 'economy_rate_normal', 'bowling_average_high', 'bowling_average_normal', 'bowling_strike_rate_high', 'bowling_strike_rate_normal', 'dot_ball_percentage_high', 'dot_ball_percentage_normal', 'boundary_percentage_high', 'boundary_percentage_normal']

Sample comparison:
 bowler_id         bowler_name  legal_balls_high  legal_balls_normal  runs_conceded_high  runs_conceded_normal  wickets_high  wickets_normal  matches_high  matches_normal  economy_rate_high  economy_rate_normal  bowling_average_high  bowling_average_normal  bowling_strike_rate_high  bowling_strike_rate_normal  dot_ball_percentage_high  dot_ball_percentage_normal  boundary_percentage_high  boundary_percentage_normal
        28       Ishant Sharma             114.0              

In [51]:
# Calculate performance changes under high pressure

# Positive values indicate improvement
bowler_comparison["economy_improvement"] = (
    bowler_comparison["economy_rate_normal"]
    - bowler_comparison["economy_rate_high"]
)

bowler_comparison["average_improvement"] = (
    bowler_comparison["bowling_average_normal"]
    - bowler_comparison["bowling_average_high"]
)

bowler_comparison["strike_rate_improvement"] = (
    bowler_comparison["bowling_strike_rate_normal"]
    - bowler_comparison["bowling_strike_rate_high"]
)

bowler_comparison["dot_ball_improvement"] = (
    bowler_comparison["dot_ball_percentage_high"]
    - bowler_comparison["dot_ball_percentage_normal"]
)

bowler_comparison["boundary_reduction"] = (
    bowler_comparison["boundary_percentage_normal"]
    - bowler_comparison["boundary_percentage_high"]
)

# Wickets per 100 legal balls
bowler_comparison["wicket_rate_high"] = (
    bowler_comparison["wickets_high"]
    / bowler_comparison["legal_balls_high"]
) * 100

bowler_comparison["wicket_rate_normal"] = (
    bowler_comparison["wickets_normal"]
    / bowler_comparison["legal_balls_normal"]
) * 100

bowler_comparison["wicket_rate_improvement"] = (
    bowler_comparison["wicket_rate_high"]
    - bowler_comparison["wicket_rate_normal"]
)

print("Performance change summary:")
print(
    bowler_comparison[
        [
            "economy_improvement",
            "average_improvement",
            "strike_rate_improvement",
            "dot_ball_improvement",
            "boundary_reduction",
            "wicket_rate_improvement"
        ]
    ].describe().round(3)
)

print("\nMissing values:")
print(
    bowler_comparison[
        [
            "economy_improvement",
            "average_improvement",
            "strike_rate_improvement",
            "dot_ball_improvement",
            "boundary_reduction",
            "wicket_rate_improvement"
        ]
    ].isna().sum()
)

Performance change summary:
       economy_improvement  average_improvement  strike_rate_improvement  \
count              156.000              155.000                  155.000   
mean                -0.688                9.709                    7.856   
std                  1.251               12.165                    7.451   
min                 -4.188              -24.742                  -16.667   
25%                 -1.405                3.969                    4.161   
50%                 -0.665                8.871                    7.583   
75%                  0.228               15.047                   10.624   
max                  2.409               70.795                   47.538   

       dot_ball_improvement  boundary_reduction  wicket_rate_improvement  
count               156.000             156.000                  156.000  
mean                 -3.940               0.211                    2.607  
std                   7.823               3.996               

In [52]:
# Identify bowlers with undefined average or strike-rate improvement

missing_metric_bowlers = bowler_comparison[
    bowler_comparison[
        ["average_improvement", "strike_rate_improvement"]
    ].isna().any(axis=1)
]

print(
    missing_metric_bowlers[
        [
            "bowler_id",
            "bowler_name",
            "legal_balls_high",
            "legal_balls_normal",
            "wickets_high",
            "wickets_normal",
            "bowling_average_high",
            "bowling_average_normal",
            "bowling_strike_rate_high",
            "bowling_strike_rate_normal"
        ]
    ].to_string(index=False)
)

 bowler_id    bowler_name  legal_balls_high  legal_balls_normal  wickets_high  wickets_normal  bowling_average_high  bowling_average_normal  bowling_strike_rate_high  bowling_strike_rate_normal
      7094 Curtis Campher             108.0               288.0           0.0             9.0                   NaN               48.888889                       NaN                        32.0


In [53]:
# Inspect high-pressure bowling performance across eligible bowlers

print("High-pressure performance distribution:")

print(
    bowler_comparison[
        [
            "economy_rate_high",
            "wicket_rate_high",
            "dot_ball_percentage_high",
            "boundary_percentage_high",
            "economy_improvement",
            "wicket_rate_improvement"
        ]
    ].describe().round(3)
)

print("\nZero high-pressure wicket bowlers:",
      (bowler_comparison["wickets_high"] == 0).sum())

print("Zero high-pressure wicket rate:",
      (bowler_comparison["wicket_rate_high"] == 0).sum())

High-pressure performance distribution:
       economy_rate_high  wicket_rate_high  dot_ball_percentage_high  \
count            156.000           156.000                   156.000   
mean               9.348             7.313                    34.467   
std                1.249             1.966                     5.051   
min                5.262             0.000                    22.807   
25%                8.576             6.042                    31.284   
50%                9.503             7.431                    34.000   
75%               10.196             8.412                    37.380   
max               12.000            13.514                    54.074   

       boundary_percentage_high  economy_improvement  wicket_rate_improvement  
count                   156.000              156.000                  156.000  
mean                     19.323               -0.688                    2.607  
std                       4.097                1.251                   

In [54]:
# Calculate percentile-based reliability scores

scoring_metrics = {
    "economy_rate_high": (30, False),
    "wicket_rate_high": (25, True),
    "dot_ball_percentage_high": (15, True),
    "boundary_percentage_high": (10, False),
    "economy_improvement": (10, True),
    "wicket_rate_improvement": (10, True)
}

for metric, (weight, higher_is_better) in scoring_metrics.items():

    percentile = bowler_comparison[metric].rank(
        pct=True,
        ascending=higher_is_better
    )

    bowler_comparison[f"{metric}_score"] = percentile * 100

    print(f"{metric}: {weight}% weight")

# Calculate weighted reliability score

bowler_comparison["reliability_score"] = sum(
    bowler_comparison[f"{metric}_score"] * weight / 100
    for metric, (weight, higher_is_better)
    in scoring_metrics.items()
)

print("\nReliability score summary:")
print(
    bowler_comparison["reliability_score"].describe().round(2)
)

print("\nMissing reliability scores:",
      bowler_comparison["reliability_score"].isna().sum())

economy_rate_high: 30% weight
wicket_rate_high: 25% weight
dot_ball_percentage_high: 15% weight
boundary_percentage_high: 10% weight
economy_improvement: 10% weight
wicket_rate_improvement: 10% weight

Reliability score summary:
count    156.00
mean      50.32
std       19.86
min       11.15
25%       35.83
50%       49.16
75%       63.62
max       92.98
Name: reliability_score, dtype: float64

Missing reliability scores: 0


In [55]:
# Rank bowlers by reliability score

bowler_comparison = bowler_comparison.sort_values(
    "reliability_score",
    ascending=False
).reset_index(drop=True)

bowler_comparison["rank"] = (
    bowler_comparison.index + 1
)

top_15_bowlers = bowler_comparison.head(15).copy()

display_columns = [
    "rank",
    "bowler_name",
    "reliability_score",
    "legal_balls_high",
    "matches_high",
    "wickets_high",
    "economy_rate_high",
    "wicket_rate_high",
    "dot_ball_percentage_high",
    "boundary_percentage_high",
    "economy_improvement",
    "wicket_rate_improvement"
]

print("Top 15 bowlers under pressure:")

print(
    top_15_bowlers[display_columns]
    .round(2)
    .to_string(index=False)
)

Top 15 bowlers under pressure:
 rank    bowler_name  reliability_score  legal_balls_high  matches_high  wickets_high  economy_rate_high  wicket_rate_high  dot_ball_percentage_high  boundary_percentage_high  economy_improvement  wicket_rate_improvement
    1  Kuldeep Yadav              92.98             514.0          62.0          51.0               7.34              9.92                     41.25                     12.84                 0.92                     5.14
    2    Shadab Khan              92.95             188.0          21.0          17.0               6.32              9.04                     40.43                      9.57                 2.06                     6.09
    3   Hasan Mahmud              92.31             125.0          13.0          13.0               7.44             10.40                     40.80                     15.20                 0.53                     7.08
    4 Shaheen Afridi              91.12             298.0          33.0          32.0

In [56]:
# Check top 15 bowlers across IPL and T20I

top_15_ids = top_15_bowlers["bowler_id"].tolist()

top_15_format_stats = (
    active_pressure_deliveries[
        active_pressure_deliveries["bowler_id"].isin(top_15_ids)
    ]
    .groupby(
        ["bowler_id", "bowler_name", "match_format"]
    )
    .agg(
        high_pressure_balls=("is_legal_ball", "sum"),
        high_pressure_matches=("match_id", "nunique"),
        wickets=("is_bowler_credited_wicket", "sum"),
        runs_conceded=("bowler_conceded_runs", "sum")
    )
    .reset_index()
)

top_15_format_stats["economy_rate"] = (
    top_15_format_stats["runs_conceded"]
    / top_15_format_stats["high_pressure_balls"]
) * 6

print("Top 15 bowlers — format-wise high-pressure performance:")

print(
    top_15_format_stats.sort_values(
        ["bowler_id", "match_format"]
    ).round(2).to_string(index=False)
)

print("\nFormat distribution:")
print(top_15_format_stats["match_format"].value_counts())

Top 15 bowlers — format-wise high-pressure performance:
 bowler_id    bowler_name match_format  high_pressure_balls  high_pressure_matches  wickets  runs_conceded  economy_rate
      1389    Sean Abbott          T20                   24                      1        1             47         11.75
      1389    Sean Abbott         T20I                  418                     20       29            677          9.72
      2382  Jamie Overton          T20                  204                     13       14            332          9.76
      2382  Jamie Overton         T20I                  293                     17       22            386          7.90
      2490 Mohammad Nawaz         T20I                 1004                     52       66           1192          7.12
      2636     Adam Zampa          T20                  180                      8       10            282          9.40
      2636     Adam Zampa         T20I                  972                     44       61      

In [57]:
# Format-wise HIGH-PRESSURE bowling performance

top_15_ids = top_15_bowlers["bowler_id"].tolist()

top_15_format_stats = (
    active_pressure_deliveries[
        (active_pressure_deliveries["bowler_id"].isin(top_15_ids))
        & (active_pressure_deliveries["pressure_group"] == "High")
    ]
    .groupby(
        ["bowler_id", "bowler_name", "match_format"]
    )
    .agg(
        high_pressure_balls=("is_legal_ball", "sum"),
        high_pressure_matches=("match_id", "nunique"),
        wickets=("is_bowler_credited_wicket", "sum"),
        runs_conceded=("bowler_conceded_runs", "sum")
    )
    .reset_index()
)

top_15_format_stats["economy_rate"] = (
    top_15_format_stats["runs_conceded"]
    / top_15_format_stats["high_pressure_balls"]
) * 6

print("Top 15 bowlers — HIGH-PRESSURE format-wise performance:")

print(
    top_15_format_stats.sort_values(
        ["bowler_id", "match_format"]
    ).round(2).to_string(index=False)
)

print("\nFormat distribution:")
print(top_15_format_stats["match_format"].value_counts())

Top 15 bowlers — HIGH-PRESSURE format-wise performance:
 bowler_id    bowler_name match_format  high_pressure_balls  high_pressure_matches  wickets  runs_conceded  economy_rate
      1389    Sean Abbott          T20                    6                      1        0             18         18.00
      1389    Sean Abbott         T20I                  152                     17       16            225          8.88
      2382  Jamie Overton          T20                   99                      8        7            148          8.97
      2382  Jamie Overton         T20I                  143                     13       15            173          7.26
      2490 Mohammad Nawaz         T20I                  156                     21       14            155          5.96
      2636     Adam Zampa          T20                   48                      6        4             84         10.50
      2636     Adam Zampa         T20I                  304                     35       30      

In [58]:
# Strict sample-size robustness test

strict_bowlers = bowler_comparison[
    (bowler_comparison["legal_balls_high"] >= 150)
    & (bowler_comparison["matches_high"] >= 10)
].copy()

# Recalculate percentile scores within the stricter eligible pool
for metric, (weight, higher_is_better) in scoring_metrics.items():

    strict_bowlers[f"{metric}_score"] = (
        strict_bowlers[metric].rank(
            pct=True,
            ascending=higher_is_better
        ) * 100
    )

strict_bowlers["strict_reliability_score"] = sum(
    strict_bowlers[f"{metric}_score"] * weight / 100
    for metric, (weight, higher_is_better) in scoring_metrics.items()
)

strict_bowlers = strict_bowlers.sort_values(
    "strict_reliability_score",
    ascending=False
).reset_index(drop=True)

strict_bowlers["strict_rank"] = strict_bowlers.index + 1

print("Strictly eligible bowlers:", len(strict_bowlers))

print("\nStrict top 10:")
print(
    strict_bowlers[
        [
            "strict_rank",
            "bowler_name",
            "strict_reliability_score",
            "legal_balls_high",
            "matches_high",
            "wickets_high"
        ]
    ].head(10).round(2).to_string(index=False)
)

Strictly eligible bowlers: 108

Strict top 10:
 strict_rank    bowler_name  strict_reliability_score  legal_balls_high  matches_high  wickets_high
           1  Kuldeep Yadav                     94.31             514.0          62.0          51.0
           2    Shadab Khan                     93.89             188.0          21.0          17.0
           3 Shaheen Afridi                     92.59             298.0          33.0          32.0
           4 Mohammad Nawaz                     90.46             156.0          21.0          14.0
           5    Abrar Ahmed                     90.42             252.0          26.0          21.0
           6  Jamie Overton                     86.53             242.0          21.0          22.0
           7    Umran Malik                     86.48             167.0          19.0          16.0
           8     Adam Zampa                     82.50             352.0          41.0          34.0
           9     Haris Rauf                     80.79

In [59]:
# Compare primary and strict rankings

primary_top10 = bowler_comparison.head(10)[
    ["bowler_id", "bowler_name", "rank"]
].copy()

strict_top10 = strict_bowlers.head(10)[
    ["bowler_id", "strict_rank"]
].copy()

ranking_comparison = primary_top10.merge(
    strict_bowlers[["bowler_id", "strict_rank"]],
    on="bowler_id",
    how="left"
)

ranking_comparison["rank_change"] = (
    ranking_comparison["rank"]
    - ranking_comparison["strict_rank"]
)

primary_ids = set(primary_top10["bowler_id"])
strict_ids = set(strict_top10["bowler_id"])

overlap = primary_ids.intersection(strict_ids)

print("Primary top 10:", len(primary_ids))
print("Strict top 10:", len(strict_ids))
print("Common bowlers:", len(overlap))
print("Top 10 overlap percentage:", len(overlap) / 10 * 100)

print("\nRanking comparison:")
print(
    ranking_comparison.sort_values("rank")[
        ["bowler_name", "rank", "strict_rank", "rank_change"]
    ].to_string(index=False)
)

print("\nNew entrants in strict top 10:")
print(
    strict_top10[
        ~strict_top10["bowler_id"].isin(primary_ids)
    ].to_string(index=False)
)

Primary top 10: 10
Strict top 10: 10
Common bowlers: 6
Top 10 overlap percentage: 60.0

Ranking comparison:
   bowler_name  rank  strict_rank  rank_change
 Kuldeep Yadav     1          1.0          0.0
   Shadab Khan     2          2.0          0.0
  Hasan Mahmud     3          NaN          NaN
Shaheen Afridi     4          3.0          1.0
   Abrar Ahmed     5          5.0          0.0
Mohammad Nawaz     6          4.0          2.0
Sufiyan Muqeem     7          NaN          NaN
Benjamin Sears     8          NaN          NaN
   Sai Kishore     9          NaN          NaN
 Jamie Overton    10          6.0          4.0

New entrants in strict top 10:
 bowler_id  strict_rank
     25048            7
      2636            8
      7364            9
      1389           10


In [60]:
# Compare both scoring methods on the SAME 108 bowlers

common_pool = bowler_comparison[
    bowler_comparison["bowler_id"].isin(
        strict_bowlers["bowler_id"]
    )
].copy()

# Rank using the original primary reliability scores
common_pool["primary_pool_rank"] = (
    common_pool["reliability_score"]
    .rank(ascending=False, method="min")
)

# Rank using strict reliability scores
common_pool = common_pool.merge(
    strict_bowlers[
        ["bowler_id", "strict_reliability_score"]
    ],
    on="bowler_id",
    how="left"
)

common_pool["strict_pool_rank"] = (
    common_pool["strict_reliability_score"]
    .rank(ascending=False, method="min")
)

# Compare top 10 from the same candidate pool
primary_same_pool = set(
    common_pool.nsmallest(10, "primary_pool_rank")["bowler_id"]
)

strict_same_pool = set(
    common_pool.nsmallest(10, "strict_pool_rank")["bowler_id"]
)

same_pool_overlap = primary_same_pool.intersection(
    strict_same_pool
)

print("Common candidate pool:", len(common_pool))
print("Primary top 10:", len(primary_same_pool))
print("Strict top 10:", len(strict_same_pool))
print("Common top 10:", len(same_pool_overlap))
print(
    "Same-pool top 10 overlap:",
    len(same_pool_overlap) / 10 * 100,
    "%"
)

print("\nSame-pool ranking comparison:")

print(
    common_pool.sort_values("primary_pool_rank")[
        [
            "bowler_name",
            "primary_pool_rank",
            "strict_pool_rank",
            "reliability_score",
            "strict_reliability_score"
        ]
    ].head(15).round(2).to_string(index=False)
)

# Spearman rank correlation
rank_correlation = common_pool[
    ["primary_pool_rank", "strict_pool_rank"]
].corr(method="spearman").iloc[0, 1]

print("\nSpearman rank correlation:", round(rank_correlation, 4))

Common candidate pool: 108
Primary top 10: 10
Strict top 10: 10
Common top 10: 9
Same-pool top 10 overlap: 90.0 %

Same-pool ranking comparison:
       bowler_name  primary_pool_rank  strict_pool_rank  reliability_score  strict_reliability_score
     Kuldeep Yadav                1.0               1.0              92.98                     94.31
       Shadab Khan                2.0               2.0              92.95                     93.89
    Shaheen Afridi                3.0               3.0              91.12                     92.59
       Abrar Ahmed                4.0               5.0              89.87                     90.42
    Mohammad Nawaz                5.0               4.0              89.81                     90.46
     Jamie Overton                6.0               6.0              85.51                     86.53
       Umran Malik                7.0               7.0              84.74                     86.48
        Adam Zampa                8.0          

In [61]:
# Create deterministic final ranking

final_bowler_rankings = bowler_comparison.sort_values(
    [
        "reliability_score",
        "legal_balls_high",
        "matches_high"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

final_bowler_rankings["final_rank"] = (
    final_bowler_rankings.index + 1
)

final_top10_bowlers = final_bowler_rankings.head(10).copy()

final_columns = [
    "final_rank",
    "bowler_name",
    "reliability_score",
    "legal_balls_high",
    "matches_high",
    "wickets_high",
    "economy_rate_high",
    "wicket_rate_high",
    "dot_ball_percentage_high",
    "boundary_percentage_high"
]

print("FINAL TOP 10 BOWLERS UNDER PRESSURE")

print(
    final_top10_bowlers[final_columns]
    .round(2)
    .to_string(index=False)
)

FINAL TOP 10 BOWLERS UNDER PRESSURE
 final_rank    bowler_name  reliability_score  legal_balls_high  matches_high  wickets_high  economy_rate_high  wicket_rate_high  dot_ball_percentage_high  boundary_percentage_high
          1  Kuldeep Yadav              92.98             514.0          62.0          51.0               7.34              9.92                     41.25                     12.84
          2    Shadab Khan              92.95             188.0          21.0          17.0               6.32              9.04                     40.43                      9.57
          3   Hasan Mahmud              92.31             125.0          13.0          13.0               7.44             10.40                     40.80                     15.20
          4 Shaheen Afridi              91.12             298.0          33.0          32.0               7.65             10.74                     40.60                     15.44
          5    Abrar Ahmed              89.87             2

In [62]:
# Save final bowler rankings
final_top10_bowlers.to_csv(
    OUTPUT_PATH + "final_top10_bowler_rankings.csv",
    index=False
)

print("Final Top 10 bowler rankings saved successfully.")

Final Top 10 bowler rankings saved successfully.


In [63]:
# Save core bowler analysis datasets

bowler_pressure_stats.to_csv(
    OUTPUT_PATH + "bowler_pressure_stats.csv",
    index=False
)

bowler_comparison.to_csv(
    OUTPUT_PATH + "bowler_comparison.csv",
    index=False
)

final_bowler_rankings.to_csv(
    OUTPUT_PATH + "primary_bowler_rankings.csv",
    index=False
)

print("Core bowler analysis files saved successfully.")

Core bowler analysis files saved successfully.


In [64]:
# Save strict-threshold bowler rankings

strict_bowler_rankings = strict_bowlers.sort_values(
    ["strict_reliability_score", "legal_balls_high", "matches_high"],
    ascending=[False, False, False]
).reset_index(drop=True)

strict_bowler_rankings["strict_rank"] = (
    strict_bowler_rankings.index + 1
)

strict_bowler_rankings.to_csv(
    OUTPUT_PATH + "strict_bowler_rankings.csv",
    index=False
)

print("Strict-threshold bowler rankings saved successfully.")
print("Eligible bowlers:", len(strict_bowler_rankings))

Strict-threshold bowler rankings saved successfully.
Eligible bowlers: 108


In [69]:
# Save format-wise high-pressure bowling performance

bowler_format_performance.to_csv(
    OUTPUT_PATH + "bowler_format_performance.csv",
    index=False
)

print("Bowler format performance saved successfully.")
print("Records:", len(bowler_format_performance))

Bowler format performance saved successfully.
Records: 1488


In [66]:
[name for name in globals() if "format" in name.lower() and "bowler" in name.lower()]

[]

In [67]:
# Format-wise high-pressure bowling performance

bowler_format_performance = (
    active_pressure_deliveries[
        active_pressure_deliveries["pressure_group"] == "High"
    ]
    .groupby(
        ["bowler_id", "bowler_name", "match_format", "season"]
    )
    .agg(
        legal_balls=("is_legal_ball", "sum"),
        runs_conceded=("bowler_conceded_runs", "sum"),
        wickets=("is_bowler_credited_wicket", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

bowler_format_performance["economy_rate"] = (
    bowler_format_performance["runs_conceded"]
    / bowler_format_performance["legal_balls"] * 6
)

bowler_format_performance.to_csv(
    OUTPUT_PATH + "bowler_format_performance.csv",
    index=False
)

print("Bowler format performance saved successfully.")
print("Records:", len(bowler_format_performance))

Bowler format performance saved successfully.
Records: 1488


In [68]:
# Save bowler scoring weights

bowler_scoring_weights = pd.DataFrame({
    "metric": [
        "High-pressure economy rate",
        "High-pressure wicket rate",
        "High-pressure dot-ball percentage",
        "High-pressure boundary percentage",
        "Economy improvement",
        "Wicket-rate improvement"
    ],
    "weight_percentage": [30, 25, 15, 10, 10, 10],
    "preferred_direction": [
        "Lower",
        "Higher",
        "Higher",
        "Lower",
        "Higher",
        "Higher"
    ]
})

bowler_scoring_weights.to_csv(
    OUTPUT_PATH + "bowler_scoring_weights.csv",
    index=False
)

print("Bowler scoring weights saved successfully.")

Bowler scoring weights saved successfully.


In [70]:
# Verify saved bowler analysis files

expected_files = [
    "final_top10_bowler_rankings.csv",
    "bowler_pressure_stats.csv",
    "bowler_comparison.csv",
    "primary_bowler_rankings.csv",
    "strict_bowler_rankings.csv",
    "bowler_format_performance.csv",
    "bowler_scoring_weights.csv"
]

for file in expected_files:
    path = OUTPUT_PATH + file
    print(f"{file}: {'EXISTS' if os.path.exists(path) else 'MISSING'}")

final_top10_bowler_rankings.csv: EXISTS
bowler_pressure_stats.csv: EXISTS
bowler_comparison.csv: EXISTS
primary_bowler_rankings.csv: EXISTS
strict_bowler_rankings.csv: EXISTS
bowler_format_performance.csv: EXISTS
bowler_scoring_weights.csv: EXISTS
